# Multi-Hazard Early Warning System — Exploratory Data Analysis & Model Training
**Project:** SIH 2026 Multi-Hazard Prediction & Cascade Risk Assessment Pipeline  
**Target Region:** Sikkim & Eastern Himalayas (Lat 27.0°–28.1°N, Lon 88.0°–89.0°E)  
**Notebook Purpose:** All-in-One Self-Contained Pipeline for Google Colab / Jupyter.

> ⚠️ **PROTOTYPE NOTICE:** Models are calibrated for disaster early warning decision support. Optimized on **PR-AUC** and **Detection Recall** rather than misleading raw accuracy.

In [ ]:
# ==============================================================================
# 0. SETUP & DEPENDENCIES (RUN IN GOOGLE COLAB)
# ==============================================================================
!pip install -q xgboost shap scikit-learn seaborn matplotlib pandas numpy joblib

import os
import sys
import time
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)
from sklearn.impute import SimpleImputer
import shap

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ Environment initialized successfully!")

## 1. Data Ingestion & 54-Column Schema Adapter
Loads the dataset. If running in Colab, you can specify the path to `final_ml_dataset_600000.csv` or upload a file. If no file is present, it automatically generates a correlated Himalayan dataset so the notebook runs immediately.

In [ ]:
# Path configuration (Modify this path if your CSV is in Google Drive or Colab storage)
CSV_PATH = "final_ml_dataset_600000.csv"
SAMPLE_ROWS = 30000  # Set to None to load all 600,000 rows

if not os.path.exists(CSV_PATH):
    # Check local relative path
    alt_path = "../../datasets2/final_ml_dataset_600000.csv"
    if os.path.exists(alt_path):
        CSV_PATH = alt_path

if os.path.exists(CSV_PATH):
    print(f"📂 Loading real dataset from: {CSV_PATH} (sampling {SAMPLE_ROWS} rows)...")
    raw_df = pd.read_csv(CSV_PATH, nrows=SAMPLE_ROWS)
    print(f"✅ Loaded {len(raw_df):,} rows with {raw_df.shape[1]} columns.")
else:
    print("⚠️ CSV not found locally — Generating 10,000-row synthetic Himalayan prototype dataset...")
    rng = np.random.default_rng(42)
    n_rows = 10000
    pos = rng.binomial(1, 0.065, size=n_rows)
    raw_df = pd.DataFrame({
        'latitude': rng.uniform(27.0, 28.1, n_rows),
        'longitude': rng.uniform(88.0, 89.0, n_rows),
        'date': pd.date_range('2020-01-01', periods=n_rows, freq='h').strftime('%Y-%m-%d'),
        'slope_deg': rng.gamma(shape=3.0, scale=8.0, size=n_rows) + (pos * 5.0),
        'aspect_deg': rng.uniform(0, 360, n_rows),
        'elevation_m': rng.uniform(300, 4500, n_rows),
        'rainfall_1d_mm': rng.exponential(scale=15.0, size=n_rows) + (pos * 40.0),
        'rainfall_3d_mm': rng.exponential(scale=35.0, size=n_rows) + (pos * 70.0),
        'rainfall_7d_mm': rng.exponential(scale=70.0, size=n_rows) + (pos * 120.0),
        'rainfall_30d_mm': rng.exponential(scale=180.0, size=n_rows) + (pos * 150.0),
        'rainfall_anomaly': rng.normal(0, 30.0, size=n_rows) + (pos * 40.0),
        'max_rainfall_intensity_mm': rng.exponential(scale=8.0, size=n_rows) + (pos * 20.0),
        'soil_moisture': np.clip(rng.beta(2, 5, size=n_rows) + (pos * 0.25), 0.05, 0.95),
        'twi': rng.normal(8.0, 3.0, size=n_rows),
        'distance_to_fault_km': rng.exponential(scale=6.0, size=n_rows),
        'distance_to_river_km': rng.exponential(scale=2.0, size=n_rows),
        'distance_to_road_km': rng.exponential(scale=3.0, size=n_rows),
        'land_cover': rng.choice(['Forest', 'Grassland', 'Agriculture', 'Built-up', 'Bare land'], size=n_rows),
        'ndvi': np.clip(rng.uniform(0.1, 0.85, size=n_rows) - (pos * 0.15), -0.1, 0.9),
        'temperature_c': rng.normal(18.0, 6.0, size=n_rows),
        'tilt_change_deg': rng.exponential(scale=0.05, size=n_rows) + (pos * 0.4),
        'landslide_occurrence': pos
    })
    print(f"✅ Generated {len(raw_df):,} synthetic rows with realistic trigger dynamics.")

# ------------------------------------------------------------------------------
# Schema Normalizer (Adapting real 54-column dataset to canonical pipeline features)
# ------------------------------------------------------------------------------
land_cover_map = {
    'Forest': 'forest', 'Grassland': 'forest', 'Agriculture': 'agricultural',
    'Built-up': 'built', 'Bare land': 'bare', 'Water': 'bare'
}

df = pd.DataFrame()
df['latitude'] = raw_df['latitude'].astype(float)
df['longitude'] = raw_df['longitude'].astype(float)
df['timestamp'] = pd.to_datetime(raw_df.get('date', raw_df.get('timestamp', '2022-01-01')))

# Rainfall Accumulations
df['rainfall_30min_mm'] = (raw_df.get('max_rainfall_intensity_mm', 0.0) * 0.5).clip(lower=0.0)
df['rainfall_3hr_mm'] = (raw_df.get('rainfall_1d_mm', 0.0) / 8.0).clip(lower=0.0)
df['rainfall_24hr_mm'] = raw_df.get('rainfall_1d_mm', raw_df.get('rainfall_24hr_mm', 0.0)).clip(lower=0.0)
df['rainfall_3day_mm'] = raw_df.get('rainfall_3d_mm', raw_df.get('rainfall_3day_mm', 0.0)).clip(lower=0.0)
df['rainfall_7day_mm'] = raw_df.get('rainfall_7d_mm', raw_df.get('rainfall_7day_mm', 0.0)).clip(lower=0.0)
df['rainfall_30d_mm'] = raw_df.get('rainfall_30d_mm', 0.0).clip(lower=0.0)
df['rainfall_intensity'] = raw_df.get('max_rainfall_intensity_mm', 0.0).clip(lower=0.0)
df['antecedent_rainfall_anomaly'] = raw_df.get('rainfall_anomaly', df['rainfall_7day_mm'] - 80.0)

# Soil & DEM Features
sm = raw_df.get('soil_moisture', raw_df.get('soil_moisture_pct', 30.0)).astype(float)
df['soil_moisture_pct'] = (sm * 100.0 if sm.max() <= 1.0 else sm).clip(0.0, 100.0)
df['soil_moisture_change_rate'] = 0.0
df['elevation_m'] = raw_df.get('elevation_m', 1500.0).clip(lower=0.0)
df['slope_deg'] = raw_df.get('slope_deg', 25.0).clip(0.0, 90.0)
df['aspect_deg'] = raw_df.get('aspect_deg', 180.0).clip(0.0, 360.0)
df['ndvi'] = raw_df.get('ndvi', 0.6).clip(-1.0, 1.0)
df['land_cover_class'] = raw_df.get('land_cover', 'Forest').map(land_cover_map).fillna('forest')

# Proximity & Geology
df['distance_to_river_km'] = raw_df.get('distance_to_river_km', 1.0).clip(lower=0.0)
df['distance_to_road_km'] = raw_df.get('distance_to_road_km', 1.0).clip(lower=0.0)
df['distance_to_fault_km'] = raw_df.get('distance_to_fault_km', 5.0).clip(lower=0.0)
df['tilt_change_deg'] = raw_df.get('tilt_change_deg', 0.0)
df['temperature_C'] = raw_df.get('temperature_c', raw_df.get('temperature_C', 18.0))
df['twi'] = raw_df.get('twi', 8.0)
df['historical_landslide_density'] = (raw_df.get('local_relief_m', 500.0) / 100.0).clip(0.0, 50.0)

# Target variable
df['landslide_occurred'] = raw_df.get('landslide_occurrence', raw_df.get('landslide_occurred', 0)).astype(int)

pos_count = int(df['landslide_occurred'].sum())
print(f"🎯 Processed Dataset: {len(df):,} records | Landslide Positives: {pos_count:,} ({pos_count/len(df):.2%})")
df.head(3)

## 2. Exploratory Data Analysis (EDA)
Visualizing class imbalance, missing value profiles, and environmental feature distributions between stable slopes and landslide events.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 1. Class Distribution
counts = df['landslide_occurred'].value_counts()
axes[0].bar(['No Landslide (0)', 'Landslide (1)'], counts.values, color=['#3b82f6', '#ef4444'], width=0.5)
axes[0].set_title(f"Target Class Distribution (Positives: {counts.get(1, 0):,})")
axes[0].set_ylabel("Record Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + (len(df)*0.01), f"{v:,} ({v/len(df):.1%})", ha='center', fontweight='bold')

# 2. Slope Distribution by Class
sns.kdeplot(data=df, x='slope_deg', hue='landslide_occurred', common_norm=False, fill=True,
            palette={0: '#3b82f6', 1: '#ef4444'}, ax=axes[1])
axes[1].set_title("Slope Gradient Distribution (°)")
axes[1].set_xlabel("Slope Angle (Degrees)")

# 3. 7-Day Rainfall Accumulation by Class
sns.kdeplot(data=df, x='rainfall_7day_mm', hue='landslide_occurred', common_norm=False, fill=True,
            palette={0: '#3b82f6', 1: '#ef4444'}, ax=axes[2])
axes[2].set_title("7-Day Cumulative Rainfall (mm)")
axes[2].set_xlabel("Rainfall (mm)")

plt.tight_layout()
plt.show()

## 3. Feature Engineering & Multi-Hazard Compound Triggers
Engineering physical trigger interaction terms:
- `slope_rainfall_trigger`: Interaction of steep terrain and prolonged 7-day rainfall saturation.
- `saturation_hazard`: Joint effect of Topographic Wetness Index (TWI) and soil moisture.
- `fault_proximity_factor`: Inverse distance weighting to active geological fault zones.

In [ ]:
# Compound Multi-Hazard Physical Triggers
df['slope_rainfall_trigger'] = (df['slope_deg'] / 45.0) * (df['rainfall_7day_mm'] / 100.0)
df['saturation_hazard'] = (df['twi'] / 10.0) * (df['soil_moisture_pct'] / 50.0)
df['fault_proximity_factor'] = 1.0 / (df['distance_to_fault_km'].clip(lower=0.0) + 0.5)

# Skewness Reduction (Log1p)
for col in ['historical_landslide_density', 'distance_to_river_km', 'distance_to_road_km']:
    df[f'{col}_log'] = np.log1p(df[col].clip(lower=0.0))

# One-Hot Encoding Land Cover
dummies = pd.get_dummies(df['land_cover_class'], prefix='lc', dtype=float)
for expected_lc in ['forest', 'bare', 'agricultural', 'built']:
    col_name = f'lc_{expected_lc}'
    if col_name not in dummies.columns:
        dummies[col_name] = 0.0

df_model = pd.concat([df.drop(columns=['land_cover_class']), dummies], axis=1)

# Select Final Input Features
meta_cols = {'latitude', 'longitude', 'timestamp', 'landslide_occurred'}
feature_names = sorted([c for c in df_model.columns if c not in meta_cols])

print(f"✅ Total Engineered Model Features: {len(feature_names)}")
print("Sample features:", feature_names[:10])

## 4. Chronological Train / Test Split (Prevent Storm Leakage)
Sorting strictly by time: earliest 80% used for training, latest 20% held out for operational test evaluation.

In [ ]:
# Chronological sort
df_sorted = df_model.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.80)

train_df = df_sorted.iloc[:split_idx]
test_df = df_sorted.iloc[split_idx:]

X_train = train_df[feature_names].fillna(train_df[feature_names].median())
y_train = train_df['landslide_occurred']

X_test = test_df[feature_names].fillna(train_df[feature_names].median())
y_test = test_df['landslide_occurred']

print(f"📊 Training Set: {X_train.shape[0]:,} rows ({int(y_train.sum()):,} landslides, {y_train.mean():.2%})")
print(f"📊 Testing Set:  {X_test.shape[0]:,} rows ({int(y_test.sum()):,} landslides, {y_test.mean():.2%})")

## 5. Model Training with Class Balancing
Training 3 baseline candidate architectures:
1. **Logistic Regression** (Interpretable linear baseline with `class_weight='balanced'`)
2. **Random Forest** (Tree ensemble with `class_weight='balanced_subsample'`)
3. **XGBoost** (Gradient boosted trees with `scale_pos_weight` ratio)

In [ ]:
scale_pos = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=4,
                                            class_weight='balanced_subsample', n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.06,
                             scale_pos_weight=scale_pos, eval_metric='aucpr', n_jobs=-1, random_state=42)
}

predictions = {}
probabilities = {}

for name, model in models.items():
    t0 = time.time()
    print(f"⚙️ Training {name}...")
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    probabilities[name] = prob
    predictions[name] = model.predict(X_test)
    print(f"   Completed in {time.time() - t0:.2f}s")

print("\n✅ All models trained successfully!")

## 6. Model Evaluation & Operational Threshold Optimization
Comparing PR-AUC, ROC-AUC, Precision, and Recall. Evaluating models both at the naive 0.50 threshold and the **optimal operational threshold ($T^*$)** that balances disaster detection with warning credibility.

In [ ]:
results = []
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for name in models:
    prob = probabilities[name]
    pr_auc = average_precision_score(y_test, prob)
    roc_auc = roc_auc_score(y_test, prob)
    
    # Standard 0.50 cutoff
    rec_std = recall_score(y_test, prob >= 0.50)
    prec_std = precision_score(y_test, prob >= 0.50, zero_division=0)
    
    # Optimal F1 threshold
    p_curve, r_curve, thres_arr = precision_recall_curve(y_test, prob)
    f1_curve = 2 * (p_curve * r_curve) / (p_curve + r_curve + 1e-10)
    best_t_idx = np.argmax(f1_curve)
    opt_t = float(thres_arr[best_t_idx]) if best_t_idx < len(thres_arr) else 0.5
    
    rec_opt = recall_score(y_test, prob >= opt_t)
    prec_opt = precision_score(y_test, prob >= opt_t, zero_division=0)
    f1_opt = f1_score(y_test, prob >= opt_t, zero_division=0)
    
    results.append({
        'Model': name,
        'PR-AUC (Primary)': pr_auc,
        'ROC-AUC': roc_auc,
        'Recall @ 0.50': rec_std,
        'Precision @ 0.50': prec_std,
        'Optimal Threshold': opt_t,
        'Recall @ Optimal': rec_opt,
        'F1 @ Optimal': f1_opt
    })
    
    # Plot PR Curve
    axes[0].plot(r_curve, p_curve, label=f"{name} (PR-AUC = {pr_auc:.3f})")
    
    # Plot ROC Curve
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[1].plot(fpr, tpr, label=f"{name} (ROC-AUC = {roc_auc:.3f})")

# Formatting Curves
axes[0].set_title("Precision-Recall Curves (Key Metric for Imbalanced Data)")
axes[0].set_xlabel("Recall (Detection Rate)")
axes[0].set_ylabel("Precision")
axes[0].legend()

axes[1].plot([0, 1], [0, 1], 'k--', label="Random Guess (0.50)")
axes[1].set_title("ROC Curves")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.show()

# Display Metrics DataFrame
res_df = pd.DataFrame(results)
res_df

## 7. Explainable AI (SHAP Global Beeswarm & Force Diagram)
Computes Shapley values to explain both overall feature importance and single-event risk drivers.

In [ ]:
# Select best tree model (XGBoost or Random Forest)
best_eval_model = models['XGBoost']

# Subsample for fast SHAP execution (Colab-safe)
X_shap = X_test.sample(min(800, len(X_test)), random_state=42)

print("🔍 Computing SHAP TreeExplainer values...")
explainer = shap.TreeExplainer(best_eval_model)
shap_values = explainer.shap_values(X_shap)

# Global Beeswarm Summary Plot
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values, X_shap, show=False)
plt.title("Global Feature Attribution — Factors Pushing Landslide Risk Higher/Lower", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

# Local Highest-Risk Event Explanation
high_risk_idx = int(np.argmax(probabilities['XGBoost']))
high_risk_sample = X_test.iloc[high_risk_idx]
single_shap = explainer.shap_values(high_risk_sample.to_frame().T)[0]

plt.figure(figsize=(14, 3.5))
shap.force_plot(
    base_value=float(explainer.expected_value),
    shap_values=single_shap,
    features=high_risk_sample,
    matplotlib=True,
    show=False
)
plt.title(f"Local Force Plot for Highest Risk Event (P = {probabilities['XGBoost'][high_risk_idx]:.1%})", pad=20)
plt.show()

## 8. Live Real-Time Risk Prediction Function (`predict_risk`)
Demonstrates the operational API function used by the Early Warning backend to classify any coordinate observation and generate cascade payloads.

In [ ]:
def predict_risk(observation_dict, model=models['Logistic Regression']):
    """Simulate live inference call."""
    row = pd.DataFrame([observation_dict])
    # Fill any missing features with training median
    for col in feature_names:
        if col not in row.columns:
            row[col] = X_train[col].median()
            
    prob = float(model.predict_proba(row[feature_names])[:, 1][0])
    
    if prob >= 0.75:
        level, color = 'VERY_HIGH', 'red'
    elif prob >= 0.55:
        level, color = 'HIGH', 'orange'
    elif prob >= 0.35:
        level, color = 'MODERATE', 'yellow'
    else:
        level, color = 'LOW', 'green'
        
    return {
        'location': {'lat': observation_dict.get('latitude', 27.35), 'lon': observation_dict.get('longitude', 88.62)},
        'risk_probability': round(prob, 4),
        'risk_level': level,
        'risk_color': color,
        'cascade_payload': {
            'trigger_river_damming': prob >= 0.55,
            'slope_deg': observation_dict.get('slope_deg', 35.0),
            'distance_to_river_km': observation_dict.get('distance_to_river_km', 0.5)
        }
    }

# Test Observation
test_obs = {
    'latitude': 27.33,
    'longitude': 88.61,
    'rainfall_24hr_mm': 140.0,
    'rainfall_7day_mm': 210.0,
    'soil_moisture_pct': 88.0,
    'slope_deg': 43.5,
    'elevation_m': 1850.0,
    'distance_to_fault_km': 0.8,
    'distance_to_river_km': 0.2
}

prediction = predict_risk(test_obs)
print("\n🔮 Live Early Warning Output:")
for k, v in prediction.items():
    print(f"  {k}: {v}")